In [2]:
import pandas as pd

df = pd.read_csv("Cti.csv", header=None, names=["ANO", "CENA", "CONTA", "VALOR"])
df.head(10)

,ANO,CENA,CONTA,VALOR
0,Ano 1,Total Cen_00001,NaN,"0,025138815"
1,Ano 1,Total Cen_00001,BAL - Total do Ativo,"10.631.127.075,26"
2,Ano 1,Total Cen_00001,BAL - Ativo Circulante,"1.795.720.872,18"
3,Ano 1,Total Cen_00001,BAL - Disponível,"1.205.264.965,52"
4,Ano 1,Total Cen_00001,BAL - Contas a Receber - SWAP,"95.680.421,92"
5,Ano 1,Total Cen_00001,BAL - Contas a Receber - Partes Relacionadas,"584.661,47"
6,Ano 1,Total Cen_00001,BAL - Contas a Receber - Clientes,"453.870.957,15"
7,Ano 1,Total Cen_00001,BAL - Estoques Diversos,"86.620,79"
8,Ano 1,Total Cen_00001,BAL - Outros Créditos,"40.233.245,33"
9,Ano 1,Total Cen_00001,BAL - Realizável a Longo Prazo,"639.023.858,23"


In [3]:
# Organizar em demonstrativos: BP (BAL), DRE e DFC (FLU)
conta = df["CONTA"].astype("string").str.strip()

BP = df[conta.str.startswith("BAL -", na=False)].copy()
DRE = df[conta.str.startswith("DRE -", na=False)].copy()
DFC = df[conta.str.startswith("FLU -", na=False)].copy()

print(f"BP  (Balanço Patrimonial): {len(BP):,} linhas | {BP['CONTA'].nunique()} contas")
print(f"DRE (Resultado):           {len(DRE):,} linhas | {DRE['CONTA'].nunique()} contas")
print(f"DFC (Fluxo de Caixa):      {len(DFC):,} linhas | {DFC['CONTA'].nunique()} contas")

BP.head(3), DRE.head(3), DFC.head(3)

BP  (Balanço Patrimonial): 468,000 linhas | 34 contas
DRE (Resultado):           201,600 linhas | 14 contas
DFC (Fluxo de Caixa):      172,800 linhas | 12 contas


(     ANO             CENA                   CONTA              VALOR
 1  Ano 1  Total Cen_00001    BAL - Total do Ativo  10.631.127.075,26
 2  Ano 1  Total Cen_00001  BAL - Ativo Circulante   1.795.720.872,18
 3  Ano 1  Total Cen_00001        BAL - Disponível   1.205.264.965,52,
       ANO             CENA           CONTA              VALOR
 44  Ano 1  Total Cen_00001   DRE - Receita   6.722.729.306,39
 45  Ano 1  Total Cen_00001  DRE - Tributos    -471.464.205,94
 46  Ano 1  Total Cen_00001    DRE - Custos  -1.361.164.977,97,
       ANO             CENA                CONTA             VALOR
 59  Ano 1  Total Cen_00001  FLU - Saldo Inicial    774.615.183,61
 60  Ano 1  Total Cen_00001        FLU - Receita  5.396.830.757,26
 61  Ano 1  Total Cen_00001       FLU - Tributos   -473.257.578,14)

In [4]:
# Separar os índices de cada demonstrativo
BP = BP.reset_index(drop=True)
DRE = DRE.reset_index(drop=True)
DFC = DFC.reset_index(drop=True)

BP.head(3), DRE.head(3), DFC.head(3)

(     ANO             CENA                   CONTA              VALOR
 0  Ano 1  Total Cen_00001    BAL - Total do Ativo  10.631.127.075,26
 1  Ano 1  Total Cen_00001  BAL - Ativo Circulante   1.795.720.872,18
 2  Ano 1  Total Cen_00001        BAL - Disponível   1.205.264.965,52,
      ANO             CENA           CONTA              VALOR
 0  Ano 1  Total Cen_00001   DRE - Receita   6.722.729.306,39
 1  Ano 1  Total Cen_00001  DRE - Tributos    -471.464.205,94
 2  Ano 1  Total Cen_00001    DRE - Custos  -1.361.164.977,97,
      ANO             CENA                CONTA             VALOR
 0  Ano 1  Total Cen_00001  FLU - Saldo Inicial    774.615.183,61
 1  Ano 1  Total Cen_00001        FLU - Receita  5.396.830.757,26
 2  Ano 1  Total Cen_00001       FLU - Tributos   -473.257.578,14)

In [5]:
# Ranking: rentabilidade, liquidez e risco por cenário
def parse_valor_br(serie):
    s = serie.astype("string").str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def pegar_conta(base, nome_conta):
    # aceita variação de espaços em "BAL -" / "BAL  -"
    m = base["CONTA"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip() == nome_conta
    out = base.loc[m, ["ANO", "CENA", "VALOR"]].copy()
    out["VALOR"] = parse_valor_br(out["VALOR"])
    return out


receita = pegar_conta(DRE, "DRE - Receita").rename(columns={"VALOR": "receita"})
resultado = pegar_conta(DRE, "DRE - Resultado Líquido").rename(columns={"VALOR": "resultado"})
ebitda = pegar_conta(DRE, "DRE - EBITDA").rename(columns={"VALOR": "ebitda"})
ativo_circ = pegar_conta(BP, "BAL - Ativo Circulante").rename(columns={"VALOR": "ativo_circ"})
passivo_circ = pegar_conta(BP, "BAL - Passivo Circulante").rename(columns={"VALOR": "passivo_circ"})
disponivel = pegar_conta(BP, "BAL - Disponível").rename(columns={"VALOR": "disponivel"})
total_ativo = pegar_conta(BP, "BAL - Total do Ativo").rename(columns={"VALOR": "total_ativo"})
total_passivo = pegar_conta(BP, "BAL - Total do Passivo").rename(columns={"VALOR": "total_passivo"})

ind = (
    receita.merge(resultado, on=["ANO", "CENA"], how="inner")
    .merge(ebitda, on=["ANO", "CENA"], how="left")
    .merge(ativo_circ, on=["ANO", "CENA"], how="left")
    .merge(passivo_circ, on=["ANO", "CENA"], how="left")
    .merge(disponivel, on=["ANO", "CENA"], how="left")
    .merge(total_ativo, on=["ANO", "CENA"], how="left")
    .merge(total_passivo, on=["ANO", "CENA"], how="left")
)

# Indicadores por cenário (média nos anos)
por_cena = (
    ind.groupby("CENA", as_index=False)
    .agg(
        receita=("receita", "mean"),
        resultado=("resultado", "mean"),
        ebitda=("ebitda", "mean"),
        ativo_circ=("ativo_circ", "mean"),
        passivo_circ=("passivo_circ", "mean"),
        disponivel=("disponivel", "mean"),
        total_ativo=("total_ativo", "mean"),
        total_passivo=("total_passivo", "mean"),
    )
)

por_cena["rentabilidade"] = por_cena["resultado"] / por_cena["receita"]  # margem líquida
por_cena["liquidez"] = por_cena["ativo_circ"] / por_cena["passivo_circ"]  # liquidez corrente
por_cena["risco"] = por_cena["total_passivo"] / por_cena["total_ativo"]  # alavancagem

# Cortes por quartil
q_rent = por_cena["rentabilidade"].quantile([0.25, 0.75])
q_liq = por_cena["liquidez"].quantile([0.25, 0.75])
q_risco = por_cena["risco"].quantile([0.25, 0.75])

por_cena["Alta rentabilidade"] = por_cena["rentabilidade"] >= q_rent[0.75]
por_cena["Retorno moderado"] = por_cena["rentabilidade"].between(q_rent[0.25], q_rent[0.75], inclusive="left")
por_cena["Alta liquidez"] = por_cena["liquidez"] >= q_liq[0.75]
por_cena["Baixa liquidez"] = por_cena["liquidez"] <= q_liq[0.25]
por_cena["Baixo risco"] = por_cena["risco"] <= q_risco[0.25]
por_cena["Alto risco"] = por_cena["risco"] >= q_risco[0.75]


def montar_selo(row):
    selos = [
        nome
        for nome in [
            "Alta rentabilidade",
            "Retorno moderado",
            "Alta liquidez",
            "Baixa liquidez",
            "Baixo risco",
            "Alto risco",
        ]
        if row[nome]
    ]
    return " | ".join(selos) if selos else "—"


por_cena["classificacao"] = por_cena.apply(montar_selo, axis=1)

Ranking = por_cena[
    [
        "CENA",
        "rentabilidade",
        "liquidez",
        "risco",
        "Alta rentabilidade",
        "Retorno moderado",
        "Alta liquidez",
        "Baixa liquidez",
        "Baixo risco",
        "Alto risco",
        "classificacao",
    ]
].sort_values(["rentabilidade", "liquidez"], ascending=False)

print("Contagem por categoria:")
for col in [
    "Alta rentabilidade",
    "Retorno moderado",
    "Alta liquidez",
    "Baixa liquidez",
    "Baixo risco",
    "Alto risco",
]:
    print(f"  {col}: {int(Ranking[col].sum()):,}")

Ranking.head(15)

Contagem por categoria:
  Alta rentabilidade: 300
  Retorno moderado: 600
  Alta liquidez: 300
  Baixa liquidez: 300
  Baixo risco: 301
  Alto risco: 302


,CENA,rentabilidade,liquidez,risco,Alta rentabilidade,Retorno moderado,Alta liquidez,Baixa liquidez,Baixo risco,Alto risco,classificacao
1127,Total Cen_01128,0.432937,2.370546,-1.0,True,False,True,False,True,False,Alta rentabilidade | Alta liquidez | Baixo risco
1126,Total Cen_01127,0.431813,2.273225,-1.0,True,False,True,False,True,False,Alta rentabilidade | Alta liquidez | Baixo risco
1125,Total Cen_01126,0.430682,2.176507,-1.0,True,False,True,False,True,False,Alta rentabilidade | Alta liquidez | Baixo risco
1124,Total Cen_01125,0.429543,2.080389,-1.0,True,False,True,False,True,False,Alta rentabilidade | Alta liquidez | Baixo risco
735,Total Cen_00736,0.429536,1.325168,-1.0,True,False,False,False,False,False,Alta rentabilidade
392,Total Cen_00393,0.42945,1.328274,-1.0,True,False,False,False,False,True,Alta rentabilidade | Alto risco
771,Total Cen_00772,0.429248,1.310432,-1.0,True,False,False,False,True,False,Alta rentabilidade | Baixo risco
560,Total Cen_00561,0.429103,1.347647,-1.0,True,False,False,False,True,False,Alta rentabilidade | Baixo risco
375,Total Cen_00376,0.429021,1.524405,-1.0,True,False,True,False,True,False,Alta rentabilidade | Alta liquidez | Baixo risco
193,Total Cen_00194,0.42901,1.342855,-1.0,True,False,False,False,False,True,Alta rentabilidade | Alto risco


In [ ]:
# Análises de Ciclo Operacional e Gestão de Capital de Giro

def _num(serie):
    s = serie.astype("string").str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def _conta(base, *nomes):
    """Soma uma ou mais contas por ANO × CENA."""
    m = base["CONTA"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip().isin(nomes)
    tmp = base.loc[m, ["ANO", "CENA", "VALOR"]].copy()
    tmp["VALOR"] = _num(tmp["VALOR"])
    return tmp.groupby(["ANO", "CENA"], as_index=False)["VALOR"].sum()


# --- Componentes do BP / DRE ---
contas_receber = _conta(
    BP,
    "BAL - Contas a Receber - Clientes",
    "BAL - Contas a Receber - Partes Relacionadas",
    "BAL - Contas a Receber - SWAP",
).rename(columns={"VALOR": "contas_receber"})

estoques = _conta(BP, "BAL - Estoques Diversos").rename(columns={"VALOR": "estoques"})
cred_trib = _conta(BP, "BAL - Créditos Tributários").rename(columns={"VALOR": "creditos_tributarios"})
fornecedores = _conta(BP, "BAL - Fornecedores").rename(columns={"VALOR": "fornecedores"})
encargos = _conta(BP, "BAL - Encargos Sociais e Trabalhistas").rename(columns={"VALOR": "encargos_sociais"})
tributos_pagar = _conta(BP, "BAL - Tributos a pagar").rename(columns={"VALOR": "tributos_a_pagar"})
disponivel = _conta(BP, "BAL - Disponível").rename(columns={"VALOR": "disponivel"})
emprestimos_cp = _conta(BP, "BAL - Empréstimos").rename(columns={"VALOR": "emprestimos_cp"})
dre_receita = _conta(DRE, "DRE - Receita").rename(columns={"VALOR": "dre_receita"})
dre_custos = _conta(DRE, "DRE - Custos").rename(columns={"VALOR": "dre_custos"})

CapGiro = (
    contas_receber.merge(estoques, on=["ANO", "CENA"], how="outer")
    .merge(cred_trib, on=["ANO", "CENA"], how="outer")
    .merge(fornecedores, on=["ANO", "CENA"], how="outer")
    .merge(encargos, on=["ANO", "CENA"], how="outer")
    .merge(tributos_pagar, on=["ANO", "CENA"], how="outer")
    .merge(disponivel, on=["ANO", "CENA"], how="outer")
    .merge(emprestimos_cp, on=["ANO", "CENA"], how="outer")
    .merge(dre_receita, on=["ANO", "CENA"], how="outer")
    .merge(dre_custos, on=["ANO", "CENA"], how="outer")
    .fillna(0)
)

# 1) NCG e Saldo de Tesouraria
# ACO = Contas a Receber + Estoques + Créditos Tributários
# PCO = Fornecedores + Encargos Sociais + Tributos a Pagar
CapGiro["ACO"] = CapGiro["contas_receber"] + CapGiro["estoques"] + CapGiro["creditos_tributarios"]
CapGiro["PCO"] = CapGiro["fornecedores"] + CapGiro["encargos_sociais"] + CapGiro["tributos_a_pagar"]
CapGiro["NCG"] = CapGiro["ACO"] - CapGiro["PCO"]
CapGiro["Saldo_Tesouraria"] = CapGiro["disponivel"] - CapGiro["emprestimos_cp"]

# 2) Prazos médios (custos entram em módulo — na DRE vêm negativos)
custos_abs = CapGiro["dre_custos"].abs()
CapGiro["PMR"] = (CapGiro["contas_receber"] / CapGiro["dre_receita"]) * 365  # dias
CapGiro["PME"] = (CapGiro["estoques"] / custos_abs) * 365                  # PMR-Estoque
CapGiro["PMP"] = (CapGiro["fornecedores"] / custos_abs) * 365
CapGiro["Ciclo_Financeiro"] = CapGiro["PMR"] + CapGiro["PME"] - CapGiro["PMP"]

# Resumo por cenário (média nos anos)
CapGiro_resumo = (
    CapGiro.groupby("CENA", as_index=False)[
        ["NCG", "Saldo_Tesouraria", "PMR", "PME", "PMP", "Ciclo_Financeiro"]
    ]
    .mean()
    .sort_values("NCG", ascending=False)
)

print("Fórmulas aplicadas:")
print("  NCG = (Contas a Receber + Estoques + Créditos Tributários) - (Fornecedores + Encargos + Tributos a Pagar)")
print("  Saldo de Tesouraria = Disponível - Empréstimos CP")
print("  PMR = (Contas a Receber / Receita) × 365")
print("  PME = (Estoques / |Custos|) × 365")
print("  PMP = (Fornecedores / |Custos|) × 365")
print("  Ciclo Financeiro = PMR + PME - PMP")
print(f"\nLinhas ANO×CENA: {len(CapGiro):,} | Cenários: {CapGiro['CENA'].nunique():,}")
print("\nAmostra CapGiro (Ano × Cenário):")
display(CapGiro[["ANO", "CENA", "NCG", "Saldo_Tesouraria", "PMR", "PME", "PMP", "Ciclo_Financeiro"]].head(10))
print("\nResumo por cenário (média):")
display(CapGiro_resumo.head(15))